**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/jones-2021"], check=True)
    os.chdir("/content/encore/entries/jones-2021")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Aery Jones et al. (2021) from dandiset 000165: linear track (run) sessions

**Paper.** Aery Jones EA, Rao A, Zilberter M, Djukic B, Bant JS, Gillespie AK, Koutsodendris N, Nelson M,
Yoon SY, Huang K, Yuan H, Gill TM, Huang Y, Frank LM (2021). Dentate gyrus and CA3 GABAergic
interneurons bidirectionally modulate signatures of internal and external drive to CA1. *Cell
Reports* 37, 110159. [doi:10.1016/j.celrep.2021.110159](https://doi.org/10.1016/j.celrep.2021.110159).
The claims file (`claims/jones-2021.yaml`) was written from the accepted author manuscript in PubMed
Central (PMC9069800) with its main supplement; Table S7 was not available to the reader.

**Dandiset.** [DANDI:000165](https://dandiarchive.org/dandiset/000165/0.211118.1526), version
0.211118.1526 (doi:10.48324/dandi.000165/0.211118.1526). The paper links the draft, not a version.
This is the only published version, and the draft has the same 572 assets and size, so the most
recent published version is pinned. Files are streamed with remfile and h5py; nothing is written to disk.

**Notebooks.** `rest_home_cage.ipynb` tests the claims on the home cage (rest) sessions;
`run_linear_track.ipynb` tests those on the linear track (run) sessions; `csd_cohort1.ipynb` tests
the CSD claim on the single-shank home cage sessions of cohort 1. Methods are in the helper
`jones2021.py`, whose docstring says where each parameter comes from.

**Common method (Method M of the claims file).** For each feature, a linear mixed model feature ~
treatment + (treatment|animal), fitted by maximum likelihood, with an F test of the treatment
coefficient (CNO minus vehicle); the model is fitted exactly from per-mouse, per-treatment sufficient
statistics (cell counts, means and sums of squares), which gives the same likelihood as fitting every
observation. The genotype test is the likelihood ratio of feature ~ treatment + (treatment|animal)
against feature ~ treatment:genotype + (treatment:genotype|animal) on the pooled data, with 9 degrees
of freedom as Table S3 gives. Two readings of the log transform are run: (a) log when the within-cell
residuals have skewness above 1 (assumption: the Methods log right-skewed features without a rule);
(b) log for every feature (the code). Significance is Holm-Bonferroni at 0.05 over the paper's family
(24 rest or 36 run comparisons per group or pair), with comparisons not computed here ranked after
those computed.

| Claim | Tested in | Status |
|---|---|---|
| fig1-tables1-expression | none | blocked: data_missing (histology not deposited) |
| figs1ef-exvivo-cno | none | blocked: data_missing (slice recordings not deposited) |
| fig2ef-csd-sr | csd | tested (cohort 1, `csd_cohort1.ipynb`) |
| fig3b-mua-during-swr | rest | tested |
| fig3c-swr-rate-bidirectional | rest | tested |
| fig3d-swr-chains | rest | tested |
| fig3e-swr-length-sst | rest | tested |
| swr-longer-100ms-sst | rest | tested |
| fig3f-swr-frequency-pv | rest | tested |
| fig3g-sw-amplitude-pv | rest | tested |
| figs2a-swr-size-pv | rest | tested |
| tables2-fast-ripples-pv | none | blocked: data_missing (LFP filtered at 300 Hz, sampled at 1 kHz) |
| tables2-detection-unchanged | rest | tested |
| tables2-rest-sg-pv | rest | tested |
| fig4d-swr-sg-power-pv | rest | tested |
| fig4e-swr-sg-coherence-pv | rest | tested |
| figs2cd-sg-predicts-sw-frequency | rest | tested |
| ca3-swr-overlap | rest | tested |
| figs3-ca3-swr-pv | rest | tested |
| figs3c-ca3-swr-rate-sst | rest | tested |
| figs3-dentate-spikes | rest | tested |
| fig5b-mua-run to figs5-theta-coherence-frequency (14 claims) | run | tested |
| running-speed-unchanged | run | tested |
| tables4-genotype-baseline | run | tested |
| tables6-sex-swr-size | rest and run | tested |
| sex-stratified-effects | rest and run | tested |
| figs6-double-like-pv, figs6-double-unlike-sst, figs6-empty-vector-control | rest and run | tested |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import jones2021 as J

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
started = time.time()

assets = J.list_assets()
lt_assets = assets[assets.kind == "LinearTrack"]
print(f"dandiset {J.DANDISET_ID} version {J.DANDISET_VERSION}: {len(lt_assets)} linear track files of {len(assets)}")
results = J.process_many(lt_assets.asset_id.tolist(), "LinearTrack", workers=4)
errors = [r for r in results if "error" in r]
print(f"processed in {(time.time() - started) / 60:.1f} min; failed: {len(errors)}")
for e in errors:
    print("  ", e)

## Data check

In [ ]:
S = J.session_table(results)
S = S[S.animal.notna()].reset_index(drop=True)
chk = pd.DataFrame([dict(session=r["meta"]["session_id"], probe=r["meta"]["probe"], lfp_min=r["lfp_range"][0],
                         lfp_max=r["lfp_range"][1], conversion=r["conv"], pos_hz=r["pos"]["pos_hz"], lfp_t0=r["pos"]["lfp_t0"], pos_t0=r["pos"]["pos_t0"],
                         pos_end_minus_lfp_end=r["pos"]["pos_t1"] - r["n_samples"] / 1000,
                         x_range=np.ptp(r["pos"]["x_range"]), y_range=np.ptp(r["pos"]["y_range"]),
                         run_min=r["run_s"] / 60, bins_1s=r["n_bins"], speed_run=r["speed_run"], speed_all=r["speed_all"],
                         zero_spikes=r["n_zero_spikes"], n_ca1_pyr=len(r["regions"]["ca1_pyr"]), n_ca3=len(r["regions"]["ca3"]),
                         n_dg=len(r["regions"]["dg"]), clock_m30=r["clock"][-30], clock_0=r["clock"][0], clock_p30=r["clock"][30],
                         track_swr=r.get("track_swr_n", 0), track_lag=r.get("track_mua_lag", np.nan),
                         track_lag_ratio=r.get("track_mua_lag_ratio", np.nan),
                         seconds=r["seconds"], maxrss_gb=r["maxrss_kb"] / 1e6)
                    for r in (results[i] for i in S.i)])
print("Probes:", chk.probe.unique())
print("Mice and sessions by group and treatment:")
print(S.groupby(["group", "treatment"]).agg(sessions=("session", "size"), mice=("animal", "nunique"),
                                             minutes=("minutes", "median")).round(1).to_string())
seq = S.assign(n=S.session.str.extract(r"(\d+)_")[0].astype(int)).sort_values(["animal", "n"]).groupby("animal").agg(
    group=("group", "first"), sequence=("treatment", lambda t: "".join("C" if x == "CNO" else "v" for x in t)),
    sessions=("n", lambda s: ",".join(map(str, s))))
print("\nTreatment sequence per mouse (v vehicle, C CNO):")
print(seq.to_string())
print("\nPer-session values:")
print(chk.drop(columns=["session", "probe"]).describe().round(3).T.to_string())
st = S.assign(start=[pd.Timestamp(results[i]["meta"]["start"]) for i in S.i]).sort_values(["animal", "start"])
gap = st.groupby("animal").start.apply(lambda t: t.diff().min().total_seconds() / 60)
print("\nShortest gap between the starts of two track sessions of one mouse (min):")
print(gap.describe().round(1).to_string())
print("Mice with two session starts less than 30 min apart:", int((gap < 30).sum()), "of", len(gap))
print("\nSessions without CA1-pyr, CA3 or DG sites:", int((chk.n_ca1_pyr == 0).sum()), int((chk.n_ca3 == 0).sum()), int((chk.n_dg == 0).sum()))
print("LFP starting_time (s) over sessions:", sorted(set(chk.lfp_t0)), "; first position timestamp (s):", sorted(set(chk.pos_t0.round(3)))[:5])


Checks of the data:

- 170 linear track files from 30 mice (8 PV-Cre, 7 SST-Cre, 7 PV-Cre/SST-Cre, 8 empty vector), all on
  the 4-shank probe, as the paper's cohort 3. Sessions last about 25 min; the Methods say 30 min.
- Every mouse has vehicle sessions first and CNO sessions after them (sequence column); the Methods say
  the treatments alternated. Treatment is confounded with session order.
- The session start times put the track sessions of each mouse minutes apart (shortest gap above), which
  cannot be right for 25 min sessions; the start times are wrong.
- The track is about 37 cm long in x (x_range), as the Methods' 36 cm track. Position is at 19.3 Hz.
- LFP values stay within about +-12500 file units on the track. As in the home cage notebook, the unit
  attribute says microvolts.

In [ ]:
# Clock of position against the LFP and the spikes: correlation over 1 s bins of the whole session between speed
# shifted by k s (positive: speed later) and CA1 log theta power, log theta/delta (1-4 Hz) power ratio, and CA1 MUA rate
scan = pd.concat([results[i]["clock_scan"].assign(session=results[i]["meta"]["session_id"]) for i in S.i])
mean_scan = scan.groupby("lag_s")[["theta", "theta_delta", "mua_ca1", "mua_theta"]].mean()
print("Mean correlation over sessions, lags -60 to +60 s (every 5 s, and -3 to +3 s):")
print(mean_scan.loc[sorted(set(range(-60, 61, 5)) | set(range(-3, 4)))].round(3).to_string())
peaks = scan.groupby("session").apply(lambda g: pd.Series({c: g.lag_s.values[np.nanargmax(g[c].values)] for c in ("theta_delta", "theta", "mua_ca1")}))
print("\nLag of each session's maximum correlation (s): sessions within 2 s of zero, of", len(peaks))
for c in ("theta_delta", "theta", "mua_ca1"):
    print(f"  {c}: {int((peaks[c].abs() <= 2).sum())}; median {peaks[c].median():.0f} s; quartiles {peaks[c].quantile(0.25):.0f} to {peaks[c].quantile(0.75):.0f} s")
z0 = scan[scan.lag_s == 0].set_index("session").theta_delta
far = scan[scan.lag_s.abs() >= 30].groupby("session").theta_delta.mean()
print(f"theta/delta correlation at zero lag: mean {z0.mean():.3f}, positive in {int((z0 > 0).sum())} of {len(z0)} sessions; "
      f"mean over |lag| >= 30 s {far.mean():.3f}; zero lag above that in {int((z0 > far.reindex(z0.index)).sum())} sessions")
print("\nSpike clock on the track: CA1 MUA cross-correlogram around SWRs detected in still periods (speed < 1 cm/s), +-2 s")
print(f"  sessions with at least 20 SWRs: {int(chk.track_lag.notna().sum())} of {len(chk)}; SWRs per session median {chk.track_swr.median():.0f}")
okl = chk.track_lag_ratio >= J.LAG_MIN_RATIO
print(f"  sessions with a peak at least {J.LAG_MIN_RATIO} times the median: {int(okl.sum())}; their lag (s):")
print(chk.track_lag[okl].describe().round(3).to_string())
print("  sessions with |lag| <= 20 ms among them:", int((chk.track_lag[okl].abs() <= 0.02).sum()))
print("Peak memory of the workers (GB):", round(chk.maxrss_gb.max(), 2), "; mean seconds per file:", round(chk.seconds.mean(), 1))


Clock checks. Over 1 s bins of the whole session, the log theta/delta power ratio at the CA1 SWR
site correlates with speed at zero lag (mean r = 0.134, positive in 141 of 170 sessions) and hardly
with speed shifted by 30 s or more (mean 0.009); 122 of 170 sessions have their maximum within 2 s of
zero lag (median 0 s). Theta power alone peaks with speed 2 to 3 s later (mean 0.125 at +3 s against
0.030 at 0 s), consistent with theta rising before running. Every file gives the LFP a starting_time of
0 s and its position timestamps start at 0 s. Position and LFP are therefore consistent with one clock
to within about 2 s, the resolution of this check, and the run periods are taken as deposited. CA1 MUA
rate also follows speed, with its maximum at +1 to +2 s (mean 0.070 at +2 s), so the spikes are on the
position clock to within a few seconds. A finer spike-clock check, CA1 MUA around SWRs in still periods
of the track, gives a clear peak in only 24 of 170 sessions (median lag -0.245 s, 5 within 20 ms), so
the sub-second spike offset on the track is not established for most sessions; run MUA (fig5b) is a
rate over 100 ms bins wholly in run, which such an offset changes only at the edges of run periods.


In [ ]:
GROUPS = ("PV", "SST", "PS", "EV")
PAIRS = (("PV", "SST"), ("PV", "PS"), ("SST", "PS"))
t0 = time.time()
TAB = J.build_tables(results, S, {**J.RUN_FEATURES, **{k: g for k, (_, g) in J.RUN_VARIANTS.items()}})
res = J.test_all(TAB, groups=GROUPS, pairs=PAIRS)
counter = {k: m for k, (m, _) in J.RUN_VARIANTS.items()}
res = J.add_decisions(res, J.RUN_FAMILY, list(J.RUN_FEATURES), counter)
print(f"Run comparisons: {len(J.RUN_FEATURES)} features (the paper's {J.RUN_FAMILY}); tests run: {len(res)} ({time.time() - t0:.0f} s)")
print("Observations per feature:", {k: len(v) for k, v in TAB.items()})
print(res[(res.reading == "a") & res.feature.isin(list(J.RUN_FEATURES))].pivot_table(index="feature", columns="test", values="p", aggfunc="first").map(lambda v: f"{v:.2g}").to_string())

def part_table(parts, r=None):
    t = J.show_parts(res if r is None else r, parts)
    print(t.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

## running-speed-unchanged

> Running speed does not differ between vehicle and CNO sessions (4.8 against 5.1 cm/s).

Paired t test across the track mice of each mouse's mean speed per treatment (mean over sessions).
Readings: (a) smoothed speed over run (above 1 cm/s); (b) over the whole session.

In [ ]:
D = pd.DataFrame([dict(animal=a, group=g, cno=int(tr == "CNO"), run=results[i]["speed_run"], whole=results[i]["speed_all"])
                  for i, a, g, tr in zip(S.i, S.animal, S.group, S.treatment)])
for col, name in (("run", "(a) run periods"), ("whole", "(b) whole session")):
    m = D.groupby(["animal", "cno"])[col].mean().unstack().dropna()
    tt = stats.ttest_rel(m[1], m[0])
    print(f"{name}: {len(m)} mice, vehicle {m[0].mean():.2f} cm/s, CNO {m[1].mean():.2f} cm/s, "
          f"t({len(m) - 1}) = {tt.statistic:.2f}, p = {tt.pvalue:.3g}; "
          f"within tolerance: {abs(m[0].mean() - 4.8) <= 0.48 and abs(m[1].mean() - 5.1) <= 0.51}")
print("Range of mouse means (a):", round(D.run.min(), 2), "to", round(D.run.max(), 2), "cm/s")

**Result: not reproduced.** Running speed during run is 4.70 cm/s on vehicle days and 5.24 cm/s on CNO days, both within tolerance of 4.8 and 5.1, but the paired t test across 30 mice is significant (t(29) = 2.69, p = 0.012), where the paper found p = 0.2. Over the whole session the difference is larger (0.96 against 1.51 cm/s, p = 8.4e-06). With vehicle days always first, this can be an effect of practice.

## fig5b-mua-run

> During movement, suppressing either class increases MUA in DG and CA3 but not in CA1.

MUA rate in 100 ms bins wholly in run, at each region's MUA site. Reading (a) logs right-skewed features; (b) logs all.

In [ ]:
part_table([("mua_ca1", "PV", "ns"), ("mua_ca1", "SST", "ns"), ("mua_ca3", "PV", "up"), ("mua_ca3", "SST", "up"),
                ("mua_dg", "PV", "up"), ("mua_dg", "SST", "up")])
print("\nVariant: spike times shifted by the session's track spike-clock lag (sessions with a clear peak only):")
part_table([("mua_ca1_aligned", "PV", "ns"), ("mua_ca1_aligned", "SST", "ns"), ("mua_ca3_aligned", "PV", "up"), ("mua_ca3_aligned", "SST", "up"),
            ("mua_dg_aligned", "PV", "up"), ("mua_dg_aligned", "SST", "up")])
print("Mice with shifted spike times, by group and treatment (0 vehicle, 1 CNO); a test needs 2 or more mice and both treatments:")
print(TAB["mua_ca1_aligned"].groupby(["group", "cno"]).animal.nunique().unstack(fill_value=0).to_string())


**Result: not reproduced.** MUA during run does not change in CA1 (p = 0.14 and 0.046, not significant after correction) and does not increase in CA3 or DG in either genotype; all four CA3 and DG coefficients are negative (p from 0.09 to 0.82). Reading (b) gives the same CA3 and DG result and makes the SST-Cre CA1 decrease significant. The variant with spike times shifted by the track lag gives no test: only 24 of 170 sessions have a clear lag, and the PV-Cre mice have such sessions only under vehicle (2 mice) and the SST-Cre mice only under CNO (2 mice), so neither group can be tested.


## fig5d-sg-power-run

> During movement, suppressing PV+ interneurons amplifies SG power in CA1 with no effect in DG and CA3, while suppressing SST+ interneurons dampens SG power in DG and CA3 with no effect in CA1; all these effects differ between genotypes.

In [ ]:
part_table([("sg_pow_ca1", "PV", "up"), ("sg_pow_ca3", "PV", "ns"), ("sg_pow_dg", "PV", "ns"),
                ("sg_pow_ca1", "SST", "ns"), ("sg_pow_ca3", "SST", "down"), ("sg_pow_dg", "SST", "down"),
                ("sg_pow_ca1", "PV vs SST", "sig"), ("sg_pow_ca3", "PV vs SST", "sig"), ("sg_pow_dg", "PV vs SST", "sig")])

**Result: not reproduced.** SST+ suppression dampens SG power in CA3 (p = 0.00064) and DG (p = 1.2e-06) and leaves CA1 unchanged (p = 0.028, not significant after correction), and PV+ suppression leaves CA3 and DG unchanged, as claimed. But PV+ suppression does not amplify SG power in CA1 (coefficient -0.10, p = 0.71), and no genotype test is significant (p = 0.19, 0.45, 0.44). Five of nine parts hold; the partial rule needs the CA1 PV-Cre increase.

## fig5e-sg-coherence-run

> During movement, suppressing PV+ interneurons enhances SG coherence between CA1 and CA3.

Readings of the bins: (a) coherence per 1 s bin (the main table); (b) one coherence per session over all
run, tested with the same LMM on session values.

In [ ]:
part_table([("sg_coh_ca3", "PV", "up")])
print("\nReported: SST-Cre and genotype test")
part_table([("sg_coh_ca3", "SST", "ns"), ("sg_coh_ca3", "PV vs SST", "ns")])
SESS = {}
for k in ("theta_coh_ca3", "sg_coh_ca3", "fg_coh_ca3", "theta_coh_dg", "sg_coh_dg", "fg_coh_dg"):
    SESS[k] = J.long_table(results, S, lambda r, k=k: [r["coh_session"].get(k, np.nan)])
res_sess = J.add_decisions(J.test_all(SESS, groups=("PV", "SST"), pairs=(("PV", "SST"),)), J.RUN_FAMILY, list(SESS))
print("\nReading (b), session coherence (decisions with these six as the computed family):")
part_table([("sg_coh_ca3", "PV", "up"), ("sg_coh_ca3", "SST", "ns")], res_sess)

**Result: not reproduced.** SG coherence between CA1 and CA3 decreases with PV+ suppression (per 1 s bin -0.034, p = 0.0024, not significant after Holm untransformed and significant logged; over the session -0.077, p = 0.00024). The claim is an increase.

## fig5f-sg-frequency-pv

> Suppressing PV+, but not SST+, interneurons reduces SG frequency during movement in CA1, CA3 and DG, and the genotypes differ.

In [ ]:
part_table([("sg_freq_ca1", "PV", "down"), ("sg_freq_ca3", "PV", "down"), ("sg_freq_dg", "PV", "down"),
                ("sg_freq_ca1", "SST", "ns"), ("sg_freq_ca3", "SST", "ns"), ("sg_freq_dg", "SST", "ns")])
print("\nGenotype tests, reported:")
part_table([("sg_freq_ca1", "PV vs SST", "sig"), ("sg_freq_ca3", "PV vs SST", "sig"), ("sg_freq_dg", "PV vs SST", "sig")])
print("\nVariant (b): one site per region:")
part_table([("sg_freq1_ca1", "PV", "down"), ("sg_freq1_ca3", "PV", "down"), ("sg_freq1_dg", "PV", "down"),
            ("sg_freq1_ca1", "SST", "ns"), ("sg_freq1_ca3", "SST", "ns"), ("sg_freq1_dg", "SST", "ns")])
print("\nMice per region with frequency values (pooled reading):",
      {f: TAB[f].groupby("group").animal.nunique().to_dict() for f in ("sg_freq_ca1", "sg_freq_ca3", "sg_freq_dg")})


**Result: not reproduced.** With peaks pooled over the sites of each region (the main reading), PV+ suppression lowers SG frequency in CA3 (p = 0.0087) and DG (p = 0.033), neither passing Holm, and not in CA1 (p = 0.44); SST+ suppression changes it nowhere (p from 0.30 to 0.75). With one site per region (variant b) the PV-Cre CA3 decrease is significant (p = 0.00099), CA1 (p = 0.30) and DG (p = 0.13) are not. The partial rule needs the CA1 decrease under either reading. No genotype test is significant (p from 0.11 to 0.998; with 5 degrees of freedom from 0.013 to 0.93). Refitted to the likelihood maximum (fit check below), the PV-Cre decreases in CA3 (p = 0.00066) and DG (p = 0.0015) pass Holm and the DG genotype test passes with 5 degrees of freedom (p = 0.00019), but CA1 still does not change (p = 0.14), so the outcome is the same. All 8 PV-Cre and 7 SST-Cre mice have frequency values in every region.


## fig5g-sg-theta-modulation

> Suppressing PV+ interneurons increases theta modulation of SG locally in DG and CA3 but not in CA1, while suppressing SST+ interneurons reduces theta modulation of SG in CA1, and these effects differ between genotypes.

In [ ]:
part_table([("sg_mi_ca3", "PV", "up"), ("sg_mi_dg", "PV", "up"), ("sg_mi_ca1", "SST", "down"), ("sg_mi_ca1", "PV", "ns"),
                ("sg_mi_ca3", "PV vs SST", "sig"), ("sg_mi_dg", "PV vs SST", "sig")])
print("\nCA1 genotype test, reported:")
part_table([("sg_mi_ca1", "PV vs SST", "ns")])

**Result: not reproduced.** Theta modulation of SG decreases with PV+ suppression in CA3 (p = 0.018) and DG (p = 0.00099, significant), the opposite of the claim, and does not decrease in CA1 with SST+ suppression (coefficient +0.02, p = 0.56). The PV-Cre CA1 effect is not significant (p = 0.11), as claimed, and the CA3 and DG genotype tests are not (p = 0.51 and 0.51).

## fig6b-fg-power-sst

> Suppressing SST+ interneurons amplifies FG power in CA1 and DG during movement, an effect specific to this class.

In [ ]:
part_table([("fg_pow_ca1", "SST", "up"), ("fg_pow_dg", "SST", "up"), ("fg_pow_ca1", "PV", "ns"), ("fg_pow_dg", "PV", "ns"),
                ("fg_pow_ca1", "PV vs SST", "sig"), ("fg_pow_dg", "PV vs SST", "sig")])

**Result: not reproduced.** FG power does not rise with SST+ suppression: it falls in CA1 (p = 0.080) and falls significantly in DG (p = 1.1e-05). PV-Cre effects are not significant (p = 0.48 and 0.0099 in CA1 and DG), and neither genotype test is (p = 0.54 and 0.69).

## figs4e-sg-fg-inverse

> In DG and CA3 of SST-Cre mice, the CNO-induced change in SG power predicts the change in FG power, animals with the greatest SG suppression showing the greatest FG enhancement.

Per SST-Cre mouse, CNO minus vehicle; (a) difference of the mouse's means (log scale where the
feature is logged under reading (a) of the log transform, and raw); (b) difference of the LMM fitted
conditional responses. Pearson correlation across mice. Expected R2 0.81 (CA3) and 0.84 (DG), tolerance 10%.

In [ ]:
rows = []
for reg in ("ca3", "dg"):
    for scale in ("raw", "log"):
        dsg, dfg = TAB[f"sg_pow_{reg}"].copy(), TAB[f"fg_pow_{reg}"].copy()
        if scale == "log":
            dsg["value"], dfg["value"] = J.log_values(dsg.value), J.log_values(dfg.value)
        for reading, fn in (("a mean difference", J.mean_changes), ("b LMM conditional", J.blup_changes)):
            x, y = fn(dsg, "SST"), fn(dfg, "SST")
            x, y = x.align(y, join="inner")
            r, p = stats.pearsonr(x, y)
            rows.append(dict(region=reg, scale=scale, reading=reading, mice=len(x), r=r, R2=r * r, p=p,
                             sg_change_range=f"{x.min():.3g} to {x.max():.3g}", fg_change_range=f"{y.min():.3g} to {y.max():.3g}"))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** Across the 7 SST-Cre mice the CNO-induced changes in SG and FG power are positively correlated, not negatively: in CA3 r = 0.92 (raw mean differences, p = 0.0034; logged r = 0.83, p = 0.020) and in DG r = 0.81 (p = 0.027; logged r = 0.67, p = 0.10). Both readings of a mouse's change give the same values. The mice with the largest SG decrease have the largest FG decrease.

## fig6e-sg-fg-ratio

> SST+ suppression decreases the SG/FG power ratio in CA1, PV+ suppression increases it throughout the hippocampus, and these effects differ between genotypes across the hippocampus.

In [ ]:
part_table([("sgfg_ca1", "SST", "down"), ("sgfg_ca1", "PV", "up"), ("sgfg_ca3", "PV", "up"), ("sgfg_dg", "PV", "up"),
                ("sgfg_ca1", "PV vs SST", "sig"), ("sgfg_ca3", "PV vs SST", "sig"), ("sgfg_dg", "PV vs SST", "sig")])

**Result: not reproduced.** PV+ suppression raises the SG/FG ratio in CA3 (p = 0.00055) and DG (p = 0.00087), as claimed, but not in CA1 (p = 0.32), and SST+ suppression does not lower it in CA1 (p = 0.14). No genotype test is significant (p = 0.17, 0.040 and 0.051). The partial rule needs the CA1 parts.

## fig6f-fg-coherence-unchanged

> FG coherence between CA1 and CA3 and between CA1 and DG does not change with either suppression.

In [ ]:
part_table([("fg_coh_ca3", "PV", "ns"), ("fg_coh_ca3", "SST", "ns"), ("fg_coh_dg", "PV", "ns"), ("fg_coh_dg", "SST", "ns")])
print("\nReading (b), session coherence:")
part_table([("fg_coh_ca3", "PV", "ns"), ("fg_coh_ca3", "SST", "ns"), ("fg_coh_dg", "PV", "ns"), ("fg_coh_dg", "SST", "ns")], res_sess)

**Result: reproduced.** FG coherence does not change between CA1 and CA3 or CA1 and DG with either suppression: per 1 s bin p from 0.20 to 0.50, and over each session p from 0.23 to 0.90, with both log readings. The coefficients are -0.02 to +0.03.

## fig6g-fg-frequency-sst

> Suppressing SST+ interneurons increases FG frequency throughout the hippocampus.

In [ ]:
part_table([("fg_freq_ca1", "SST", "up"), ("fg_freq_ca3", "SST", "up"), ("fg_freq_dg", "SST", "up")])
print("\nVariant (b): one site per region:")
part_table([("fg_freq1_ca1", "SST", "up"), ("fg_freq1_ca3", "SST", "up"), ("fg_freq1_dg", "SST", "up")])
print("\nMice per region with frequency values (pooled reading):",
      {f: TAB[f].groupby("group").animal.nunique().to_dict() for f in ("fg_freq_ca1", "fg_freq_ca3", "fg_freq_dg")})


**Result: partially reproduced.** With peaks pooled over each region's sites, FG frequency rises with SST+ suppression in CA3 (p = 1.4e-08) and DG (p = 2.2e-11) but not significantly in CA1 (p = 0.013, short of Holm). With one site per region the main fit gives DG only in this run (p = 2.2e-08; CA1 p = 0.81, CA3 0.24); in the run the verifier reviewed it gave no region (DG p = 0.33), because for these features L-BFGS-B stops short of the maximum at points that differ from run to run (fit check below). Refitted to the maximum, both readings give CA3 and DG (pooled p = 2.8e-07 and 2.4e-09; one site p = 3.5e-06 and 7.1e-09) and not CA1 (p = 0.0093 and 0.58). All 7 SST-Cre mice are included, and pooled FG frequency rises in each of them in CA3 and DG (t test of the per-mouse changes p = 0.0031 and 0.0015). The outcome rests on the pooled reading of frequency and on fitting to the maximum, assumptions.


## fig6h-fg-theta-modulation-sst

> Suppressing SST+ interneurons increases theta modulation of FG in CA1, an effect specific to SST+ interneurons.

In [ ]:
part_table([("fg_mi_ca1", "SST", "up"), ("fg_mi_ca1", "PV vs SST", "sig")])
print("\nPV-Cre, reported with its sign:")
part_table([("fg_mi_ca1", "PV", "down")])

**Result: not reproduced.** Theta modulation of FG in CA1 does not increase significantly with SST+ suppression (coefficient 0.073, p = 0.085), and the genotype test is not significant (p = 0.15). The PV-Cre effect is negative (-0.14, p = 0.067).

## figs4h-fg-theta-modulation-pv

> Suppressing PV+ interneurons dampens theta modulation of FG during movement.

In [ ]:
part_table([("fg_mi_ca3", "PV", "down"), ("fg_mi_dg", "PV", "down")])
print("\nCA1, reported with its sign:")
part_table([("fg_mi_ca1", "PV", "down")])

**Result: partially reproduced.** PV+ suppression dampens theta modulation of FG in DG (p = 2.1e-06) and in CA3 with p = 0.0098, which does not pass Holm; in CA1 the effect is negative too (p = 0.067). The claim holds in one of the two required regions.

## figs5-theta-power-ca1

> Suppressing either PV+ or SST+ interneurons increases theta power in CA1 but not in CA3 or DG during movement.

In [ ]:
part_table([("theta_pow_ca1", "PV", "up"), ("theta_pow_ca1", "SST", "up"), ("theta_pow_ca3", "PV", "ns"),
                ("theta_pow_ca3", "SST", "ns"), ("theta_pow_dg", "PV", "ns"), ("theta_pow_dg", "SST", "ns")])

**Result: not reproduced.** Theta power in CA1 does not rise with either suppression (PV-Cre -0.34 on the log scale, p = 0.34; SST-Cre -0.07, p = 0.80). In DG it falls with SST+ suppression (p = 0.0015, significant) and with PV+ suppression (p = 0.0019, not significant after Holm).

## figs5-theta-coherence-frequency

> Neither interneuron type modifies theta coherence or theta frequency, except for an increase in theta frequency in DG of PV-Cre mice.

In [ ]:
parts = [("theta_freq_dg", "PV", "up")] + [(f, g, "ns") for f in ("theta_coh_ca3", "theta_coh_dg", "theta_freq_ca1", "theta_freq_ca3", "theta_freq_dg")
                                           for g in ("PV", "SST") if not (f == "theta_freq_dg" and g == "PV")]
part_table(parts)
print("\nCoherence reading (b), session coherence:")
part_table([(f, g, "ns") for f in ("theta_coh_ca3", "theta_coh_dg") for g in ("PV", "SST")], res_sess)
print("\nFrequency variant (b): one site per region:")
part_table([("theta_freq1_dg", "PV", "up")] + [(f, g, "ns") for f in ("theta_freq1_ca1", "theta_freq1_ca3", "theta_freq1_dg")
                                                for g in ("PV", "SST") if not (f == "theta_freq1_dg" and g == "PV")])


**Result: reproduced, under the pooled reading of frequency (an assumption).** With peaks pooled over each region's sites, theta frequency rises in DG with PV+ suppression (+0.040 log units, p = 0.00049), the stated exception, and none of the other nine tests is significant (theta coherence p from 0.015 to 0.47; theta frequency p from 0.0071 to 0.55, the smallest, PV-Cre CA3 and CA1, short of Holm). With one site per region (variant b) PV-Cre CA3 theta frequency is significant (p = 0.0011), which the claim excludes, so the outcome rests on the pooled reading; Method M averages a region over its sites, and Table S3's counts of data points imply peaks from several sites. Refitted to the maximum, the pooled results are the same (fit check). DG theta frequency rises in all 8 PV-Cre mice (t test of the per-mouse changes p = 0.014); referred to the number of mice, the F test gives p = 0.010, which would not pass Holm over 36. The per-peak p-value rests on the per-peak degrees of freedom of the paper's code.


## tables4-genotype-baseline

> During vehicle treatment, PV-Cre mice differ from SST-Cre mice in DG and CA3 SG frequency (lower) and SG/FG ratios (higher).

Vehicle sessions of PV-Cre and SST-Cre mice; LMM feature ~ genotype + (genotype|animal) with an F test
(the same fit as the treatment test, with genotype in place of treatment); group means are means over
mice of each mouse's mean. Holm over the 36 run features.

In [ ]:
rows = []
for f in J.RUN_FEATURES:
    d = TAB[f]
    d = d[d.group.isin(("PV", "SST")) & d.session.isin(S[S.treatment == "Vehicle"].session)].copy()
    d["cno"] = (d.group == "SST").astype(int)          # coefficient is SST minus PV
    log = J.within_skew(d) > J.SKEW_LOG
    dd = d.copy()
    if log:
        dd["value"] = J.log_values(dd.value)
    r = J.treatment_test(J.cell_stats(dd, "value"))
    m = d.groupby(["group", "animal"]).value.mean().groupby(level=0).mean()
    rows.append(dict(feature=f, log=log, sst_minus_pv=r["coef"], p=r["p"], PV=m["PV"], SST=m["SST"]))
G4 = pd.DataFrame(rows)
G4["holm_36"] = J.holm(G4.p.values, 36)
rows = []
for f in ("sg_freq1_ca3", "sg_freq1_dg"):
    d = TAB[f]
    d = d[d.group.isin(("PV", "SST")) & d.session.isin(S[S.treatment == "Vehicle"].session)].copy()
    d["cno"] = (d.group == "SST").astype(int)
    r = J.treatment_test(J.cell_stats(d, "value"))
    m = d.groupby(["group", "animal"]).value.mean().groupby(level=0).mean()
    rows.append(dict(feature=f, sst_minus_pv=r["coef"], p=r["p"], PV=m["PV"], SST=m["SST"],
                     mice=d.animal.nunique(), significant_at_0_05_over_36=r["p"] < 0.05 / 36))
print("Variant (b) of frequency, one site per region (untransformed):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
show = G4[G4.feature.isin(["sg_freq_ca3", "sg_freq_dg", "sgfg_ca3", "sgfg_dg"])].copy()
exp = {"sg_freq_ca3": (40.54, 41.9), "sg_freq_dg": (41.06, 42.4), "sgfg_ca3": (6.2, 3.99), "sgfg_dg": (5.16, 3.4)}
show["expected_PV"] = show.feature.map(lambda f: exp[f][0])
show["expected_SST"] = show.feature.map(lambda f: exp[f][1])
print(show.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nOther run features with a vehicle genotype difference under Holm:", G4[G4.holm_36 & ~G4.feature.isin(list(exp))].feature.tolist())

**Result: reproduced, under the pooled reading of frequency (an assumption).** In vehicle sessions PV-Cre mice have lower SG frequency in CA3 (39.66 against 40.95 Hz, p = 0.00056) and DG (40.16 against 41.48 Hz, p = 0.00034), and higher SG/FG ratios in CA3 (6.08 against 3.89, p = 0.00054) and DG (5.09 against 3.27, p = 0.00026), each passing Holm over 36. All eight group means are within tolerance of Table S4. With one site per region (variant b, untransformed) neither frequency difference is significant (CA3 p = 0.31, DG p = 0.44, 15 mice); logged, as the pooled values are, CA3 is significant (p = 0.00052) and DG is not (p = 0.0063) (fit check). The outcome rests on the pooled reading. The refit leaves the four p-values as they are. Referred to the number of mice they would be 0.0029 to 0.0044, and Welch t tests of the per-mouse means give 0.0051 to 0.0070, none below 0.05/36.


## tables6-sex-swr-size and sex-stratified-effects (run features)

Run features for the two sex claims: the vehicle sex difference (all track mice) and, per group,
the treatment by sex likelihood ratio test.

In [ ]:
rows, rows2 = [], []
Sv = S[S.treatment == "Vehicle"]
for f in J.RUN_FEATURES:
    d = TAB[f][TAB[f].session.isin(Sv.session)].copy()
    d["cno"] = (d.sex == "F").astype(int)
    log = J.within_skew(d) > J.SKEW_LOG
    if log:
        d["value"] = J.log_values(d.value)
    r = J.treatment_test(J.cell_stats(d, "value"))
    rows.append(dict(feature=f, log=log, female_minus_male=r["coef"], p=r["p"]))
    for g in ("PV", "SST", "PS"):
        d = TAB[f][TAB[f].group == g].copy()
        if J.within_skew(d) > J.SKEW_LOG:
            d["value"] = J.log_values(d.value)
        csF, csM = J.cell_stats(d[d.sex == "F"], "value"), J.cell_stats(d[d.sex == "M"], "value")
        if csF.animal.nunique() < 2 or csM.animal.nunique() < 2:
            continue
        gt = J.genotype_test(csF, csM)
        rows2.append(dict(group=g, feature=f, coef_F=gt["coef_a"], coef_M=gt["coef_b"], p=gt["p_df9"],
                          mice_F=csF.animal.nunique(), mice_M=csM.animal.nunique()))
SEXR = pd.DataFrame(rows)
SEXR["holm_36"] = J.holm(SEXR.p.values, 36)
SEXR["p<0.05"] = SEXR.p < 0.05
print("Vehicle sex differences, run features:")
print(SEXR.sort_values("p").to_string(index=False, float_format=lambda v: f"{v:.3g}"))
SXI = pd.DataFrame(rows2)
SXI["holm"] = False
for g, d in SXI.groupby("group"):
    SXI.loc[d.index, "holm"] = J.holm(d.p.values, len(d))
print("\nTreatment by sex interaction, run features (Holm per group over the 36):")
print(SXI[SXI.holm | SXI.feature.isin(["fg_pow_ca1", "fg_pow_ca3", "fg_pow_dg", "mua_ca3", "mua_dg"])]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Significant interactions per group:", SXI.groupby("group").holm.sum().to_dict())
print("Mice by group and sex on the track (an interaction needs 2 or more of each sex):")
print(S.groupby(["group", "sex"]).animal.nunique().unstack().to_string())


**Result (run part).** No run feature differs by sex in vehicle sessions after Holm over 36 (smallest p = 0.011, DG theta power); five have p < 0.05 uncorrected. Two run features have a significant sex interaction, PV-Cre CA1 FG frequency (p = 0.00014) and SST-Cre CA1-CA3 theta coherence (p = 0.0012), neither a named exception; the named FG power and MUA interactions are not significant (p from 0.021 to 0.85). The P/S group has 6 female and 1 male track mice, so its interaction is not tested and the claim's 'either or both' MUA exception is untested for P/S. With the rest part, tables6-sex-swr-size is not reproduced (SWR size does not differ, p = 0.52) and sex-stratified-effects is not reproduced (no named exception holds).


## Session order (cohort 3)

Every track mouse had its three vehicle sessions before its three CNO sessions (data check). If the
run features drift with session order, the drift appears as a treatment effect. Within the vehicle
sessions alone, where treatment cannot act, the LMM compares the third vehicle session with the
first (feature ~ session + (session|animal), F test), per group, under reading (a) of the log
transform; the sign is compared with that of the treatment effect.

In [ ]:
S["order"] = S.session.str.extract(r"(\d+)_")[0].astype(int)
S["rank"] = S.groupby("animal").order.rank()
rows = []
for f in J.RUN_FEATURES:
    for g in GROUPS:
        d = TAB[f][TAB[f].group == g]
        d = d.merge(S[["session", "rank"]], on="session")
        d = d[d["rank"].isin((1, 3))].assign(cno=lambda x: (x["rank"] == 3).astype(int))
        if d.animal.nunique() < 2 or d.cno.nunique() < 2:
            continue
        lg = bool(res[(res.feature == f) & (res.reading == "a")].log.iloc[0])
        if lg:
            d = d.assign(value=J.log_values(d.value))
        t = J.treatment_test(J.cell_stats(d, "value"))
        tr = res[(res.feature == f) & (res.reading == "a") & (res.test == g)].iloc[0]
        rows.append(dict(feature=f, group=g, veh3_minus_veh1=t["coef"], p=t["p"], cno_coef=tr.coef, cno_p=tr.p,
                         same_sign=np.sign(t["coef"]) == np.sign(tr.coef), mice=t["n_animals"]))
ORD = pd.DataFrame(rows)
ORD["holm"] = False
for g, dd in ORD.groupby("group"):
    ORD.loc[dd.index, "holm"] = J.holm(dd.p.values, J.RUN_FAMILY)
print("Vehicle session 3 against vehicle session 1, per group: features with a significant drift (Holm over 36):")
print(ORD[ORD.holm].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nDrifts with p < 0.05 per group:", ORD[ORD.p < 0.05].groupby("group").size().to_dict())
print("Share of features whose drift has the sign of the treatment effect, per group:",
      ORD.groupby("group").same_sign.mean().round(2).to_dict())
ev_sig = res[(res.test == "EV") & (res.reading == "a") & res.significant & res.feature.isin(list(J.RUN_FEATURES))].feature
print("\nEmpty vector features that change with CNO, and their drift within vehicle sessions:")
print(ORD[(ORD.group == "EV") & ORD.feature.isin(ev_sig)].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
m = pd.DataFrame([dict(animal=a, group=g, rank=rk, speed=results[i]["speed_run"]) for i, a, g, rk in zip(S.i, S.animal, S.group, S["rank"])])
print("\nRun speed (cm/s) by session rank, mean over mice (ranks 1-3 vehicle, 4-6 CNO):")
print(m.groupby(["group", "rank"]).speed.mean().unstack().round(2).to_string())


**Result (session order).** Within the vehicle sessions alone, 13 run features drift significantly from the first to the third vehicle session (Holm over 36 per group), among them DG theta and FG power in PV-Cre mice and CA1, CA3 and DG theta frequency in empty vector mice; 61 to 92% of the drifts per group have the sign of the group's treatment effect. Of the nine run features that change with CNO in empty vector mice, DG theta frequency also drifts within the vehicle sessions (p = 0.00021); the other eight do not (p from 0.115 to 0.93). Run speed rises from the first to the third vehicle session in every group (empty vector 4.28 to 5.39, PV-Cre 4.52 to 4.97, SST-Cre 4.43 to 4.55, P/S 5.14 to 5.75 cm/s). Session order can thus explain part of the CNO effects, the speed difference among them, but this check does not explain most of the empty vector changes.


## figs6 claims (run features)

Double suppression (PS) against PV-Cre and SST-Cre, and the empty vector group, over the 36 run features.

In [ ]:
a = res[(res.reading == "a") & res.feature.isin(list(J.RUN_FEATURES))]
tab = pd.DataFrame({
    "PS coef": a[a.test == "PS"].set_index("feature").coef, "PS p": a[a.test == "PS"].set_index("feature").p,
    "PS sig": a[a.test == "PS"].set_index("feature").significant,
    "PV vs PS p": a[a.test == "PV vs PS"].set_index("feature").p, "PV vs PS sig": a[a.test == "PV vs PS"].set_index("feature").significant,
    "PV coef": a[a.test == "PV"].set_index("feature").coef,
    "SST vs PS p": a[a.test == "SST vs PS"].set_index("feature").p, "SST vs PS sig": a[a.test == "SST vs PS"].set_index("feature").significant,
    "SST coef": a[a.test == "SST"].set_index("feature").coef,
    "EV coef": a[a.test == "EV"].set_index("feature").coef, "EV p": a[a.test == "EV"].set_index("feature").p,
    "EV sig": a[a.test == "EV"].set_index("feature").significant})
print("Reading (a):")
print(tab.to_string(float_format=lambda v: f"{v:.3g}"))
b = res[(res.reading == "b") & res.feature.isin(list(J.RUN_FEATURES))]
for t in ("PV vs PS", "SST vs PS", "EV", "PS"):
    print(f"Significant, reading (a) / (b), {t}:", a[(a.test == t) & a.significant].feature.tolist(), "/",
          b[(b.test == t) & b.significant].feature.tolist())
print("Mice: PS", S[S.group == "PS"].animal.nunique(), " EV", S[S.group == "EV"].animal.nunique())

**Result (run part).** Double suppression against PV-Cre: of 36 run features only DG theta power differs; theta modulation of SG in CA3 and DG, the claimed exceptions, does not (p = 0.60 and 0.52). Double suppression against SST-Cre: only DG MUA differs (reading a); none of the five run tests the claim needs to differ is significant (CA1 FG theta modulation p = 0.41, DG FG power 0.98, CA3 FG power 0.93, DG SG frequency 1.0, CA1 SG/FG ratio 0.28). The P/S treatment raises FG frequency in CA3 (p = 8.5e-07) and DG (p = 1.8e-08) but not CA1 (p = 0.72), and does not change CA1 FG power (p = 0.81). Empty vector mice change with CNO in nine run features (DG theta, SG and FG power and FG frequency among them). With the rest part: figs6-double-like-pv is not reproduced (the exceptions fail, though only one of 58 features differs), figs6-double-unlike-sst is not reproduced, and figs6-empty-vector-control is not reproduced.


## Fit check (per-peak features)

The frequency features have one observation per oscillation peak, up to millions per group, and their F tests
use the residual degrees of freedom (n - 2), as the code does. Two checks of these results. First, every model
is refitted with a Nelder-Mead search continuing from the best of the three L-BFGS-B starts (`polish=True`):
with this many observations the likelihood is large, its finite-difference gradient is noisy, and L-BFGS-B
can stop short of the maximum. Second, for the frequency features the fitted random-effect standard deviations,
the same F statistic referred to the number of mice (F(1, mice - 1)), and a t test of the per-mouse mean change.

In [ ]:
t0 = time.time()
res_pol = J.add_decisions(J.test_all(TAB, groups=GROUPS, pairs=PAIRS, polish=True), J.RUN_FAMILY, list(J.RUN_FEATURES), counter)
cmp = res.merge(res_pol[["feature", "test", "reading", "coef", "p", "significant"]], on=["feature", "test", "reading"],
                suffixes=("", "_pol"))
ratio = np.abs(np.log10(cmp.p.clip(lower=1e-300)) - np.log10(cmp.p_pol.clip(lower=1e-300)))
print(f"Refitted {len(cmp)} tests in {time.time() - t0:.0f} s. p changes by more than a factor of 2 in "
      f"{int((ratio > np.log10(2)).sum())} (treatment tests {int(((ratio > np.log10(2)) & ~cmp.test.str.contains(' vs ')).sum())}, "
      f"genotype tests {int(((ratio > np.log10(2)) & cmp.test.str.contains(' vs ')).sum())}); genotype tests with p = 1 "
      f"(the full model fitted below the reduced one): {int((cmp.test.str.contains(' vs ') & (cmp.p > 0.9999)).sum())} before, "
      f"{int((cmp.test.str.contains(' vs ') & (cmp.p_pol > 0.9999)).sum())} after")
print("\nTests whose Holm decision changes with the refit:")
print(cmp[cmp.significant != cmp.significant_pol][["feature", "test", "reading", "coef", "p", "significant", "coef_pol", "p_pol",
      "significant_pol"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
for title, parts in (("fig5f-sg-frequency-pv", [("sg_freq_ca1", "PV", "down"), ("sg_freq_ca3", "PV", "down"), ("sg_freq_dg", "PV", "down"),
                                                 ("sg_freq_ca1", "SST", "ns"), ("sg_freq_ca3", "SST", "ns"), ("sg_freq_dg", "SST", "ns"),
                                                 ("sg_freq_ca1", "PV vs SST", "sig"), ("sg_freq_ca3", "PV vs SST", "sig"), ("sg_freq_dg", "PV vs SST", "sig"),
                                                 ("sg_freq1_ca1", "PV", "down"), ("sg_freq1_ca3", "PV", "down"), ("sg_freq1_dg", "PV", "down")]),
                     ("fig6g-fg-frequency-sst", [("fg_freq_ca1", "SST", "up"), ("fg_freq_ca3", "SST", "up"), ("fg_freq_dg", "SST", "up"),
                                                  ("fg_freq1_ca1", "SST", "up"), ("fg_freq1_ca3", "SST", "up"), ("fg_freq1_dg", "SST", "up")]),
                     ("figs5-theta-coherence-frequency, frequency parts", [("theta_freq_dg", "PV", "up")] + [(f, g, "ns") for f in ("theta_freq_ca1", "theta_freq_ca3", "theta_freq_dg")
                                                  for g in ("PV", "SST") if not (f == "theta_freq_dg" and g == "PV")])):
    print(f"\n{title}, refitted (reading a):")
    t = J.show_parts(res_pol, parts)
    print(t[t.reading == "a"].drop(columns=["reading"]).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

In [ ]:
rows = []
for f in [k for k in TAB if "freq" in k]:
    d = TAB[f].assign(value=J.log_values(TAB[f].value))
    for g in ("PV", "SST", "PS"):
        dd = d[d.group == g]
        r = J.fit_lmm(J.cell_stats(dd, "value"), polish=True)
        th = r["theta"]
        L = np.array([[np.exp(th[1]), 0], [th[2], np.exp(th[3])]])
        G = L @ L.T
        mc = dd.groupby(["animal", "cno"]).value.mean().unstack()
        diff = (mc[1] - mc[0]).dropna()
        F = r["beta"][1] ** 2 / r["cov"][1, 1]
        rows.append(dict(feature=f, group=g, coef=r["beta"][1], se=np.sqrt(r["cov"][1, 1]), slope_sd=np.sqrt(G[1, 1]),
                         intercept_sd=np.sqrt(G[0, 0]), resid_sd=np.exp(th[0]), p_n=stats.f.sf(F, 1, r["n"] - 2),
                         p_mice=stats.f.sf(F, 1, len(diff) - 1), mice=len(diff), mice_up=int((diff > 0).sum()),
                         mouse_change=diff.mean(), p_mouse_t=stats.ttest_1samp(diff, 0).pvalue))
FC = pd.DataFrame(rows)
print("Frequency features (log), refitted: random-effect SDs, the F test with n - 2 and with mice - 1 denominator df,")
print("and a one-sample t test of the per-mouse change (mean CNO minus mean vehicle):")
print(FC.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
rows = []
for f in ("sg_freq_ca3", "sg_freq_dg", "sgfg_ca3", "sgfg_dg", "sg_freq1_ca3", "sg_freq1_dg"):
    d = TAB[f]
    d = d[d.group.isin(("PV", "SST")) & d.session.isin(S[S.treatment == "Vehicle"].session)].copy()
    d["cno"] = (d.group == "SST").astype(int)          # coefficient is SST minus PV
    if J.within_skew(d) > J.SKEW_LOG:
        d["value"] = J.log_values(d.value)
    r0 = J.treatment_test(J.cell_stats(d, "value"))
    r1 = J.treatment_test(J.cell_stats(d, "value"), polish=True)
    m = d.groupby(["group", "animal"]).value.mean()
    w = stats.ttest_ind(m["SST"], m["PV"], equal_var=False)
    rows.append(dict(feature=f, sst_minus_pv=r1["coef"], p=r0["p"], p_refit=r1["p"],
                     p_refit_mice=stats.f.sf(r1["F"], 1, d.animal.nunique() - 2), p_welch_mouse_means=w.pvalue,
                     mice=d.animal.nunique()))
print("\ntables4-genotype-baseline, vehicle sessions, SST-Cre minus PV-Cre: main fit, refit, refit with mice - 2 df,")
print("and a Welch t test of the per-mouse means:")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result (fit check).** Refitted to the likelihood maximum, 87 of 660 run tests change p by more than a factor of 2 (35 treatment tests, 52 genotype tests), and the genotype tests with p = 1, where the full model had been fitted below the reduced one, fall from 25 to 6. The Holm decision changes in 12 tests (20 rows with the two log readings): the PV-Cre SG frequency decreases in CA3 and DG become significant, the one-site FG frequency rise of SST-Cre mice becomes significant in CA3, the empty vector CA3 FG frequency change becomes significant, four MUA genotype tests change, and four other tests of the one-site variant change. No claim's outcome changes: fig5f-sg-frequency-pv still lacks the CA1 decrease, fig6g-fg-frequency-sst holds in CA3 and DG under both frequency readings, the frequency parts of figs5-theta-coherence-frequency and the tables4 tests do not change, and the figs6 and fig5b outcomes do not depend on the changed tests. The main fits vary between runs for the per-peak features (the one-site DG FG frequency of SST-Cre mice gave p = 2.2e-08 in this run and 0.33 in the run the verifier reviewed); the refit gave the same values in both runs made of this notebook. The fitted random-slope SDs of the frequency features are 0.005 to 0.04 log units, against residual SDs of 0.18 to 0.30. Referred to the number of mice, or tested on per-mouse means, the significant frequency effects have p from about 0.001 to 0.02. They hold per mouse in direction (for example DG FG frequency rises in 7 of 7 SST-Cre mice), but with mice as the unit only the DG FG frequency rises of SST-Cre and P/S mice (p = 0.00099 and 0.00098) would pass the first Holm step over 36. The rest notebook's fits were not refitted: its run time is close to the 60 minute limit.

## Summary

| Claim | Outcome |
|---|---|
| running-speed-unchanged | not reproduced |
| fig5b-mua-run | not reproduced |
| fig5d-sg-power-run | not reproduced |
| fig5e-sg-coherence-run | not reproduced |
| fig5f-sg-frequency-pv | not reproduced |
| fig5g-sg-theta-modulation | not reproduced |
| fig6b-fg-power-sst | not reproduced |
| figs4e-sg-fg-inverse | not reproduced |
| fig6e-sg-fg-ratio | not reproduced |
| fig6f-fg-coherence-unchanged | reproduced |
| fig6g-fg-frequency-sst | partially reproduced (assumed pooled frequency and refit) |
| fig6h-fg-theta-modulation-sst | not reproduced |
| figs4h-fg-theta-modulation-pv | partially reproduced |
| figs5-theta-power-ca1 | not reproduced |
| figs5-theta-coherence-frequency | reproduced (assumed pooled frequency) |
| tables4-genotype-baseline | reproduced (assumed pooled frequency) |
| tables6-sex-swr-size | not reproduced (with the rest notebook) |
| sex-stratified-effects | not reproduced (with the rest notebook) |
| figs6-double-like-pv, figs6-double-unlike-sst, figs6-empty-vector-control | not reproduced (with the rest notebook) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: coherence with the CA1 SWR site as the CA1 reference; models fitted by L-BFGS-B from three starts, which for the per-peak features can stop short of the maximum (a refit to the maximum is shown and changes no outcome); frequency pooled over the sites of each region (one site per region as a variant); the 1 s bins wholly in run; the genotype test fitted from cell statistics; the log transform under two readings.

What was found about the dataset: every mouse ran its vehicle sessions before its CNO sessions, so treatment is confounded with session order, and the empty vector mice change with CNO in nine of 36 run features, of which one also drifts within the vehicle sessions; the session start times are wrong (sessions of a mouse start minutes apart); sessions last 25 min, not 30.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")